In [1]:
import re
import time
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import pandas as pd
import requests

# Константы целевого сайта
BASE_URL = "https://webscraper.io"
START_URL = "https://webscraper.io/test-sites/e-commerce/static/computers/laptops"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    )
}

# Константы для функции fetch
MAX_RETRIES = 3
TIMEOUT = 15
RETRY_DELAY = 15

In [2]:
def fetch(url: str) -> str | None:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)

            if r.status_code == 200:
                r.encoding = r.apparent_encoding
                return r.text

            if r.status_code == 404:
                return None

            print(f"попытка {attempt}: HTTP {r.status_code}")
        except requests.RequestException as e:
            print(f"попытка {attempt}: {e}")

        if attempt < MAX_RETRIES:
            time.sleep(RETRY_DELAY * attempt)

    print(f"не удалось загрузить {url}")
    return None

In [3]:
def parse_catalog_page(html_text, base_url):
    soup = BeautifulSoup(html_text, "html.parser")

    cards = soup.select("div.product-wrapper")
    records = []

    for card in cards:
        # Название
        title_tag = card.select_one("a.title")
        title = title_tag.get("title", "").strip() if title_tag else ""
        if not title and title_tag:
            title = title_tag.get_text(strip=True)

        # Ссылка
        rel_link = title_tag.get("href", "") if title_tag else ""
        abs_link = urljoin(base_url, rel_link)

        # Цена
        price_tag = card.select_one("h4.price")
        price = None
        if price_tag:
            raw_price = re.sub(r"[^\d.]", "", price_tag.get_text(strip=True))
            price = float(raw_price) if raw_price else None

        # Рейтинг
        rating_tag = card.select_one("p[data-rating]")
        rating = int(rating_tag.get("data-rating")) if rating_tag else 0

        # Отзывы
        reviews_tag = card.select_one("div.ratings p.review-count")
        reviews_count = 0
        if reviews_tag:
            rev_clean = re.sub(r"[^\d]", "", reviews_tag.get_text(strip=True))
            reviews_count = int(rev_clean) if rev_clean else 0

        # Описание
        desc_tag = card.select_one("p.description")
        description = desc_tag.get_text(strip=True) if desc_tag else ""

        records.append({
            "title": title,
            "price": price,
            "rating": rating,
            "reviews_count": reviews_count,
            "description": description,
            "url": abs_link,
        })

    return records

In [4]:
num_pages = 5

urls_to_scrape = [START_URL] + [f"{START_URL}?page={i}" for i in range(2, num_pages + 1)]

raw_dataset = []
for p_url in urls_to_scrape:
    html_data = fetch(p_url)

    if html_data is not None:
        items = parse_catalog_page(html_data, BASE_URL)
        raw_dataset.extend(items)
    else:
        print(f"Пропуск страницы {p_url} (нет данных)")

In [5]:
df = pd.DataFrame(raw_dataset)
df

,title,price,rating,reviews_count,description,url
0,Packard 255 G2,416.99,2,2,"15.6"", AMD E2-3800 1.3GHz, 4GB, 500GB, Windows...",https://webscraper.io/test-sites/e-commerce/st...
1,Aspire E1-510,306.99,3,2,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",https://webscraper.io/test-sites/e-commerce/st...
2,ThinkPad T540p,1178.99,1,2,"15.6"", Core i5-4200M, 4GB, 500GB, Win7 Pro 64bit",https://webscraper.io/test-sites/e-commerce/st...
3,ProBook,739.99,4,8,"14"", Core i5 2.6GHz, 4GB, 500GB, Win7 Pro 64bit",https://webscraper.io/test-sites/e-commerce/st...
4,ThinkPad X240,1311.99,3,12,"12.5"", Core i5-4300U, 8GB, 240GB SSD, Win7 Pro...",https://webscraper.io/test-sites/e-commerce/st...
5,Aspire E1-572G,581.99,1,2,"15.6"", Core i5-4200U, 8GB, 1TB, Radeon R7 M265...",https://webscraper.io/test-sites/e-commerce/st...
6,ThinkPad Yoga,1033.99,2,13,"12.5"" Touch, Core i3-4010U, 4GB, 500GB + 16GB ...",https://webscraper.io/test-sites/e-commerce/st...
7,Pavilion,609.99,1,4,"15.6"", Core i5-4200U, 6GB, 750GB, Windows 8.1",https://webscraper.io/test-sites/e-commerce/st...
8,Inspiron 15,745.99,3,12,"Moon Silver, 15.6"", Core i7-4510U, 8GB, 1TB, R...",https://webscraper.io/test-sites/e-commerce/st...
9,Dell XPS 13,1281.99,3,4,"13.3"" Touch, Core i5-4210U, 8GB, 128GB SSD, Wi...",https://webscraper.io/test-sites/e-commerce/st...


In [6]:
#  Очистка строковых полей от краевых пробелов
for col in ["title", "description", "url"]:
    df[col] = df[col].astype(str).str.strip()

# Удаление дубликатов
initial_count = len(df)
df = df.drop_duplicates(subset=["title", "description"])
print(f"Удалено дубликатов: {initial_count - len(df)}")

# Сохранение результата в CSV
output_csv = "laptops.csv"
df.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"Файл сохранен: {output_csv}")

Удалено дубликатов: 0
Файл сохранен: laptops.csv


| Критерий | Код | no-code |
|-----------|------|---------|
| Удобство первоначальной настройки | Необходимо открыть DevTools в браузере, вручную найти теги `<article>`, `<h3>` и вписать их в код | Не нужно разбираться в коде страницы, всё распознаётся автоматически |
| Скорость настройки | 20–30 минут | 3–5 минут |
| Скорость сбора | Мгновенно, скрипт забирает HTML и парсит его | 15–30 сек. Программа запускает виртуальный браузер, грузит информацию, формирует файл |
| Удобство поддержки (сайт обновился) | Если на сайте изменился класс кнопки или тег, придётся править код | Если на сайте изменился класс кнопки или тег, то визуальный алгоритм сбивается |
| Достоинства | Скорость сбора, отсутствие ограничений: нет лимитов на количество собранных строк или частоту запусков, можно преобразовывать данные в удобные форматы (проведение типов, маппинг) | Auto-detect, визуальный интерфейс: настройка происходит через клики по нужным элементам |
| Недостатки | Нужно писать код, самостоятельно искать теги нужных элементов | Лимиты, невозможно привести строку к типам, убрать скрытые неразрывные пробелы (`\xa0`), скомбинировать несколько полей или сделать маппинг. Возможна некорректная выгрузка в CSV |